
================================================================

**Author:** Carlos Stalin Saritama Atopo  
**Email:** cssaritama@gmail.com  
**Area of Specialization:** Analytics, Advanced Analytics, and Artificial Intelligence  
**Date:** August 2025 

===============================================================

# 01 — Data Collection

This notebook demonstrates how to collect market data for **AlphaTrader-ML**.

**Goals**
- Define tickers universe
- Download historical OHLCV (synthetic or via APIs)
- Save unified raw dataset (parquet/csv)

> **Note:** To allow reviewers to run this notebook without API keys, we default to **synthetic data**.
> Live data sections are included and can be enabled if keys are available.

In [ ]:
# Imports
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)  # reproducibility


## Configure Universe and Date Range

In [ ]:
# Universe and dates
tickers = ['AAPL','MSFT','AMZN','GOOGL','META','NFLX','TSLA','NVDA']
start_date = pd.Timestamp('2018-01-01')
end_date = pd.Timestamp('2024-12-31')

date_index = pd.date_range(start_date, end_date, freq='B')  # business days
len(date_index), date_index[:3], date_index[-3:]

## Generate Synthetic OHLCV
We simulate geometric Brownian motion price paths with realistic volatility and random volume.

In [ ]:
def simulate_prices(n_steps, mu=0.10, sigma=0.25, s0=100.0):
    dt = 1/252
    shocks = np.random.normal((mu-0.5*sigma**2)*dt, sigma*np.sqrt(dt), size=n_steps)
    prices = s0 * np.exp(np.cumsum(shocks))
    return prices

frames = []
for t in tickers:
    s0 = np.random.uniform(20, 400)
    prices = simulate_prices(len(date_index), mu=np.random.uniform(0.05,0.15), sigma=np.random.uniform(0.15,0.4), s0=s0)
    df = pd.DataFrame({
        'date': date_index,
        'ticker': t,
        'close': prices,
    })
    spread = np.random.uniform(0.005, 0.02, size=len(df)) * df['close'].values
    df['open'] = df['close'] * (1 + np.random.normal(0, 0.002, size=len(df)))
    df['high'] = df[['open','close']].max(axis=1) + spread
    df['low']  = df[['open','close']].min(axis=1) - spread
    df['volume'] = (np.random.lognormal(mean=12, sigma=0.5, size=len(df))).astype(int)
    frames.append(df[['date','ticker','open','high','low','close','volume']])

raw = pd.concat(frames).sort_values(['ticker','date']).reset_index(drop=True)
raw.head()

## Save Raw Dataset

In [ ]:
raw_path = RAW_DIR / 'ohlcv_synthetic.parquet'
raw.to_parquet(raw_path, index=False)
raw_path, raw.shape

## (Optional) Live Data via `yfinance`

In [ ]:
# Optional live data (commented out to keep notebook offline-friendly)
# import yfinance as yf
# live_frames = []
# for t in tickers:
#     df = yf.download(t, start=str(start_date.date()), end=str(end_date.date()))
#     df.reset_index(inplace=True)
#     df['ticker'] = t
#     df.rename(columns={'Date':'date','Open':'open','High':'high','Low':'low','Close':'close','Volume':'volume'}, inplace=True)
#     live_frames.append(df[['date','ticker','open','high','low','close','volume']])
# live = pd.concat(live_frames).sort_values(['ticker','date'])
# live.to_parquet(RAW_DIR / 'ohlcv_yf.parquet', index=False)
# live.head()